# Day 018 — Exercise 5: TokenAwareChatbot

**Goal:** Implement `TokenAwareChatbot` — a class that wraps `ollama.chat` and automatically tracks token usage with `UsageTracker`. One real Ollama call is made in the checks.

In [ ]:
import ollama

## Provided: All Prior Functions

In [ ]:
def extract_usage(response: dict) -> dict:
    return {
        'input_tokens':  response.get('prompt_eval_count', 0),
        'output_tokens': response.get('eval_count', 0),
        'duration_ms':   response.get('eval_duration', 0) // 1_000_000,
    }

def tokens_per_second(response: dict) -> float:
    duration_s = response.get('eval_duration', 0) / 1e9
    if duration_s == 0:
        return 0.0
    return response.get('eval_count', 0) / duration_s

class UsageTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.call_count = 0

    @property
    def total_tokens(self) -> int:
        return self.total_input + self.total_output

    def record(self, response: dict) -> None:
        usage = extract_usage(response)
        self.total_input  += usage['input_tokens']
        self.total_output += usage['output_tokens']
        self.call_count   += 1

PRICES = {
    'gpt-4o':            {'input': 2.50,  'output': 10.00},
    'claude-3-5-sonnet': {'input': 3.00,  'output': 15.00},
    'gemini-1.5-pro':    {'input': 1.25,  'output': 5.00},
}

def cost_estimate(input_tokens, output_tokens, model='gpt-4o'):
    if model not in PRICES:
        raise ValueError(f'Unknown model {model!r}. Choose from: {list(PRICES)}')
    p = PRICES[model]
    input_cost  = input_tokens  * p['input']  / 1_000_000
    output_cost = output_tokens * p['output'] / 1_000_000
    return {
        'model':       model,
        'input_cost':  round(input_cost,  6),
        'output_cost': round(output_cost, 6),
        'total_cost':  round(input_cost + output_cost, 6),
    }


## Your Implementation

In [ ]:
class TokenAwareChatbot:
    """
    Wraps ollama.chat with automatic usage tracking.

    Maintains a conversation history and records token usage
    for every call via UsageTracker.
    """

    def __init__(self, model='llama3.2',
                 system_prompt='You are a helpful assistant.'):
        # TODO: set self.model, create self.tracker = UsageTracker()
        # TODO: set self._history = [{'role': 'system', 'content': system_prompt}]
        # TODO: set self._last_response = None
        pass

    def chat(self, user_input: str) -> str:
        """
        Send a message and return the reply text.
        Records token usage internally via self.tracker.
        Stores the raw Ollama response in self._last_response.
        """
        # TODO: append {'role': 'user', 'content': user_input} to self._history
        # TODO: call ollama.chat(model=self.model, messages=self._history)
        # TODO: store response in self._last_response
        # TODO: call self.tracker.record(response)
        # TODO: extract reply = response['message']['content']
        # TODO: append {'role': 'assistant', 'content': reply} to self._history
        # TODO: return reply
        pass

    def summary(self) -> str:
        """
        Return a formatted string with accumulated session stats.
        Must include the word 'tok' or 'token' and the total count.
        """
        # TODO: format self.tracker fields into a readable string
        pass


## Check Your Work

In [ ]:
import io, sys

def _run_checks():
    total = 5
    passed = 0

    # Check 1: all required names defined
    try:
        for name in ('extract_usage', 'tokens_per_second',
                     'UsageTracker', 'cost_estimate', 'TokenAwareChatbot'):
            assert name in globals(), f'{name} not defined'
        passed += 1; print('✅ Check 1: all required functions and classes defined')
    except Exception as e:
        print(f'❌ Check 1: {e}')

    # Check 2: chat() returns a non-empty string (one real Ollama call)
    try:
        bot = TokenAwareChatbot(model='llama3.2')
        reply = bot.chat('Reply with only the word yes.')
        assert isinstance(reply, str) and len(reply) > 0, \
            f'expected non-empty string, got {reply!r}'
        passed += 1; print('✅ Check 2: chat() returns a non-empty string')
    except Exception as e:
        print(f'❌ Check 2: chat() — {e}')

    # Check 3: tracker records usage
    try:
        assert bot.tracker.call_count == 1, \
            f'expected call_count 1, got {bot.tracker.call_count}'
        assert bot.tracker.total_tokens > 0, \
            f'expected total_tokens > 0, got {bot.tracker.total_tokens}'
        passed += 1; print('✅ Check 3: tracker records usage after chat()')
    except Exception as e:
        print(f'❌ Check 3: tracker — {e}')

    # Check 4: summary() mentions token counts
    try:
        s = bot.summary()
        assert isinstance(s, str) and len(s) > 0
        assert any(w in s.lower() for w in ('token', 'tok', str(bot.tracker.total_tokens))), \
            f'summary() should mention token counts, got: {s!r}'
        passed += 1; print('✅ Check 4: summary() returns a string with token info')
    except Exception as e:
        print(f'❌ Check 4: summary() — {e}')

    # Check 5: cost_estimate works with tracker totals (no Ollama call)
    try:
        est = cost_estimate(bot.tracker.total_input, bot.tracker.total_output, 'gpt-4o')
        assert 'total_cost' in est and est['total_cost'] >= 0
        passed += 1; print('✅ Check 5: cost_estimate works with tracker totals')
    except Exception as e:
        print(f'❌ Check 5: cost_estimate integration — {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')

_run_checks()


## Solution

<details>
<summary>Click to reveal</summary>

```python
class TokenAwareChatbot:
    def __init__(self, model='llama3.2',
                 system_prompt='You are a helpful assistant.'):
        self.model = model
        self.tracker = UsageTracker()
        self._history = [{'role': 'system', 'content': system_prompt}]
        self._last_response = None

    def chat(self, user_input: str) -> str:
        self._history.append({'role': 'user', 'content': user_input})
        response = ollama.chat(model=self.model, messages=self._history)
        self._last_response = response
        self.tracker.record(response)
        reply = response['message']['content']
        self._history.append({'role': 'assistant', 'content': reply})
        return reply

    def summary(self) -> str:
        return (
            f'Calls: {self.tracker.call_count} | '
            f'Input: {self.tracker.total_input} tok | '
            f'Output: {self.tracker.total_output} tok | '
            f'Total: {self.tracker.total_tokens} tok'
        )
```

</details>